# R-GCN node prediction on the F1 KGs

Binary classification of `RaceEntry` nodes: did the entry score championship points
(`hasRacePoints`, 31.3% positive)? The same pipeline runs on all three KGs (`adhoc` /
`pure_sulo` / `hybrid_sulo`), with the same split, seeds, tuning budget and evaluation code,
so differences between them come from the KG construction alone.

The shared logic lives in `f1_node_data.py` (masking, split, features), `node_prediction/models.py`
and `node_prediction/gnn_tools.py` (training, tuning, evaluation). This notebook and its
GraphSAGE counterpart differ only in the model.

In [ ]:
# Must run before CUDA is initialised: avoids fragmentation OOMs across a long sweep.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "f1_population.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import torch

from f1_node_data import build_kg_data, load_population, race_entry_labels, stratified_split
from node_prediction.gnn_tools import (
    BATCH_SIZE, cleanup, run_sequential_optimization_with_verification, set_seed, train_model,
    evaluate_test_set,
)
from node_prediction.models import RGCNModel

KGS = ["adhoc", "pure_sulo", "hybrid_sulo"]
OUT_DIR = ROOT / "outputs" / "f1" / "node_prediction" / "rgcn"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(torch.__version__, "| CUDA:", torch.cuda.is_available())

## Data

`build_kg_data` renders each KG from `data/f1/assertions.csv` **without** the label and the
race outcomes (`hasFinishPosition`, `hasStatus`, `hasLapsCompleted`), for every `RaceEntry`.
Masking happens before rendering, so Pure SULO's intermediate-node paths disappear completely,
and where a start and finish position were one merged triple, the start position survives.
Labels come from the assertions, never from a graph.

The 80/10/10 split is stratified by label and computed once on IRIs, so all KGs share it.
Node features never read IRI text: an instance is described by its `rdf:type` set, a class node
by being that class, a literal by its datatype and (standardised) value or, for strings, by a
learnable embedding of the string. The degree feature is computed on the masked graph.

In [ ]:
entities, assertions = load_population(ROOT / "data" / "f1")
labels = race_entry_labels(assertions)
split = stratified_split(labels, seed=42)

kg_data = {kg: build_kg_data(kg, entities, assertions, split) for kg in KGS}

pd.DataFrame({
    kg: {
        "nodes": d.data.num_nodes,
        "edges (fwd + inv)": d.data.edge_index.size(1),
        "relation types (fwd + inv)": d.num_relations,
        "feature keys": len(d.vocabulary),
        "train / val / test": f"{len(d.train)} / {len(d.val)} / {len(d.test)}",
        "positive rate (train)": round(d.data.y[d.train].mean().item(), 3),
    }
    for kg, d in kg_data.items()
}).T

## Tuning constants

Every value below must be set, with a source, before anything is trained: the grids are
not carried over from the CAC phase. The same values are used for every KG.
R-GCN's basis decomposition is tuned over the absolute basis counts of Schlichtkrull et al., plus no decomposition (`None`), keeping only counts ≤ each KG's relation count R (forward + inverse edge types): Pure SULO (R = 24) gets {None, 2, 10, 20}.

In [ ]:
# Hidden dims: Hamilton et al. 2017 use 256-d outputs at every depth and 512/1024 pooling dims;
# Schlichtkrull et al. 2018 use 16 hidden units for entity classification.
#   https://arxiv.org/abs/1706.02216
#   https://arxiv.org/abs/1703.06103
GRID_1A_DIM = [32, 64, 128, 256, 512]

# (layers, neighbors per hop): (25,) / (25, 10) = Hamilton et al. 2017 (K=2, S1=25, S2=10; K>2 gave
# 0-5% more); 2 layers = Schlichtkrull et al. 2018; (15, 10, 5) = PyG's GraphSAGE ogbn-products example.
#   https://arxiv.org/abs/1706.02216
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/2.3.0/examples/ogbn_products_sage.py
GRID_1B_DEPTH = [(1, (25,)), (2, (25, 10)), (3, (15, 10, 5))]

# Learning rate: Hamilton et al. 2017 searched {1e-2, 1e-3, 1e-4} (supervised); Schlichtkrull et al. 2018
# and PyG's R-GCN example use 1e-2.
#   https://arxiv.org/abs/1706.02216
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/master/examples/rgcn_link_pred.py
GRID_2B_LR = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

# Weight decay (Adam): Schlichtkrull et al. 2018 searched L2 in {0, 5e-4}; GraphGym (You et al. 2020)
# defaults to 5e-4.
#   https://arxiv.org/abs/1703.06103
#   https://github.com/pyg-team/pytorch_geometric/blob/master/torch_geometric/graphgym/config.py
GRID_2C_WD = [0, 1e-5, 1e-4, 5e-4, 1e-3]

# Defaults while earlier phases run: (2, (25, 10)) = Hamilton et al. 2017; lr 1e-3 is in Hamilton et
# al.'s grid. wd 1e-4 has no direct literature value (between R-GCN's 0 and 5e-4).
#   https://arxiv.org/abs/1706.02216
DEFAULT_DEPTH = (2, (25, 10))
DEFAULT_LR = 1e-3
DEFAULT_WD = 1e-4

# R-GCN bases: Schlichtkrull et al. 2018, B in {0,10,20,30,40} for entity classification
# (B=0 = no decomposition = None here) and B=2 for link prediction. Capped at R.
#   https://arxiv.org/abs/1703.06103
GRID_BASES = [2, 10, 20, 30, 40]
DEFAULT_BASES = 10        # largest grid value <= R in every KG (Pure SULO: R = 24)

# Max epochs: GraphGym design space {100, 200, 400} (You et al. 2020, Section 4). Patience: no
# literature value (engineering choice); the final retrain uses best_epoch + 1 regardless.
#   https://arxiv.org/abs/2011.08843
EPOCHS = 100
PATIENCE = 10

# Node input embedding size: the GraphSAGE reference implementation recommends learned identity
# features of 64-256 dims (README, --identity_dim).
#   https://github.com/williamleif/GraphSAGE#readme
FEATURE_DIM = 256

# Dropout: PyG's R-GCN example uses p=0.2 between the RGCNConv layers; the original
# R-GCN paper uses no dropout for entity classification.
#   https://github.com/pyg-team/pytorch_geometric/blob/master/examples/rgcn_link_pred.py
DROPOUT = 0.2

TUNING_SEED = 0
FINAL_SEEDS = [0, 1, 2, 3, 4]   # final retrain + test, reported as mean ± std

missing = [name for name, value in dict(globals()).items() if name.isupper() and value is None]
assert not missing, f"Set these before training: {missing}"

In [ ]:
def model_factory(d):
    """(hidden_list, bases) -> fresh model for one KG; everything KG-specific is bound here."""
    def build(hidden_list, bases):
        return RGCNModel(
            num_keys=len(d.vocabulary), feature_dim=FEATURE_DIM, hidden_channels_list=hidden_list,
            num_relations=d.num_relations, num_bases=bases, dropout=DROPOUT,
        )
    return build

def bases_grid(d):
    return [None] + [b for b in GRID_BASES if b <= d.num_relations]

def default_bases(d):
    return DEFAULT_BASES

## Pipeline check: shuffled labels

Trains the default configuration (smallest grid dim) on each real graph with the labels permuted across all
labelled entries. The test ROC-AUC must stay at chance (≈ 0.5): otherwise the target vector
reaches the model's inputs somewhere in the pipeline. (Leakage of the *true* outcome through
graph facts is handled by the masking above and checked in `tests/test_f1_node_data.py`.)

In [ ]:
control = {}
for kg, d in kg_data.items():
    labelled = torch.cat([d.train, d.val, d.test])
    shuffled = d.data.clone()
    shuffled.y[labelled] = d.data.y[labelled][torch.randperm(len(labelled), generator=torch.Generator().manual_seed(0))]

    set_seed(TUNING_SEED)
    model = model_factory(d)([GRID_1A_DIM[0]] * (DEFAULT_DEPTH[0] - 1), default_bases(d))
    model, _ = train_model(model, shuffled, d.train, d.val, epochs=EPOCHS, batch_size=BATCH_SIZE,
                           num_neighbors=DEFAULT_DEPTH[1], lr=DEFAULT_LR, weight_decay=DEFAULT_WD, patience=PATIENCE)
    control[kg] = evaluate_test_set(model, shuffled, d.test, num_neighbors=DEFAULT_DEPTH[1])["roc_auc"]
    model.cpu()
    del model, shuffled
    cleanup()

control

## Sequential tuning and seeded final evaluation

Per KG: greedy search (dim → depth/neighbors → bases → lr → wd), a 2×2 verification grid
over the top dims × top regularisation combos, all seeded with `TUNING_SEED`; then the winner
is retrained on train+val for `best_epoch + 1` epochs and tested once per seed in `FINAL_SEEDS`.
Validation and test passes sample neighbourhoods with a fixed seed, so every configuration and
KG is scored on identically sampled subgraphs.

In [ ]:
tuning = {}
for kg, d in kg_data.items():
    tuning[kg] = run_sequential_optimization_with_verification(
        model_factory=model_factory(d), graph_data=d.data,
        train_indices=d.train, val_indices=d.val, test_indices=d.test,
        schema_label=kg, save_dir=f"{OUT_DIR / kg}/", csv_path=OUT_DIR / f"{kg}_sequential_tuning_results.csv",
        grid_dim=GRID_1A_DIM, grid_depth=GRID_1B_DEPTH, grid_bases=bases_grid(d), default_bases=default_bases(d),
        grid_lr=GRID_2B_LR, grid_wd=GRID_2C_WD,
        default_depth=DEFAULT_DEPTH, default_lr=DEFAULT_LR, default_wd=DEFAULT_WD,
        epochs=EPOCHS, patience=PATIENCE, seed=TUNING_SEED, final_seeds=FINAL_SEEDS,
    )
    cleanup()

## Results

Test metrics of the final model per KG, mean ± std over `FINAL_SEEDS`.

In [ ]:
final = pd.concat(
    df[df["phase"] == "4_Final_Test_Evaluation"].assign(kg=kg) for kg, df in tuning.items()
)
metrics = ["test_roc_auc", "test_pr_auc", "test_f1", "test_accuracy", "test_class0_f1", "test_class1_f1"]
summary = final.groupby("kg")[metrics].agg(["mean", "std"]).round(4)
summary.to_csv(OUT_DIR / "final_test_summary.csv")
summary